# Import packages and libraries

In [1]:
import ase
import numpy as np
import matplotlib as plt
import os 
import sys 

print(f'ASE version : {ase.__version__}')
print(f'Numpy : {np.__version__}')
print(f'MatPlotLib : {plt.__version__}')

# Build Structure
from ase import atoms
from ase.build import bulk
from ase.build import make_supercell

# Nearest-Neigbour Analysis
from ase.data import atomic_masses, chemical_symbols
from ase.neighborlist import neighbor_list

# Visualise
from ase.visualize import view

#write and read file
from ase.io import read, write

# Auxilliary
from collections import Counter

# Directory
os.makedirs('structures/bulk_structures', exist_ok=True)

ASE version : 3.27.0
Numpy : 2.4.3
MatPlotLib : 3.10.8


# Build Structure

![Material Project NiO file](/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/SNieenshot2026-04-20%at%12.18.29%PM.png)

In [2]:
atom_name= 'Ni'

In [3]:
from ase.spacegroup import crystal
import numpy as np

# 1. Define the unit cell based on the JSON parameters
a,b, c = 4.19, 4.19, 4.19
alpha, beta, gamma = 90, 90, 90

# 2. Build the stoichiometric unit cell
# Unlike Hastelloy, we don't start with a pure metal; 
# we start with the specific oxide symmetry (Spacegroup 167)
unit_cell = crystal(
    symbols=[atom_name, 'O'],
    basis=[(0, 0, 0), (0.5, 0.5, 0.5)],
    spacegroup=225,
    cellpar=[a, b, c, alpha, beta, gamma]
)

# 3. Nieate the 5x5x5 supercell 
# This replicates the 30-atom unit cell 125 times
supercell = unit_cell * (5, 5, 5)

# 4. Check the results
print(f"Alloy structure: NiO")
print(f"Total atoms: {len(supercell)}") # Result: 3750 atoms

Alloy structure: NiO
Total atoms: 1000


In [4]:
view(supercell, viewer='x3d')

In [5]:
magmoms = []

for i, atom in enumerate(supercell):
    if atom.symbol == 'Ni':
        # Alternating +2.0 and -2.0 for Ni2+ ions
        if i % 2 == 0:
            magmoms.append(2.0)
        else:
            magmoms.append(-2.0)
    else:
        # Oxygen is non-magnetic
        magmoms.append(0.0)

supercell.set_initial_magnetic_moments(magmoms)
# Physics Check:
print(f"Net Magnetization: {sum(supercell.get_initial_magnetic_moments())}") # Should be 0.0
print(f"Absolute Moment per Ni: {abs(supercell.get_initial_magnetic_moments()[0])}") # Should be 2.0

Net Magnetization: 0.0
Absolute Moment per Ni: 2.0


### Write out the pure metal-oxide file

In [6]:
# For .XYZ file
supercell_xyz_path = f'structures/bulk_structures/{atom_name}_oxide_supercell.xyz'
write(supercell_xyz_path, supercell)

# For .lammps file
MASSES = {
    1: (26.9815, 'Al'),
    2: (10.8110, 'B'),
    3: (12.0110, 'C'),
    4: (51.9961, 'Cr'),
    5: (55.8450, 'Fe'),
    6: (95.9600, 'Mo'),
    7: (58.6934, 'Ni'),
    8: (15.9990, 'O'),   # O inserted here — before H
    9: ( 1.0080, 'H'),
}

elem_to_type = {el: t for t, (m, el) in MASSES.items()}

L    = supercell.cell.lengths()
pos  = supercell.get_positions()
syms = supercell.get_chemical_symbols()

lammps_path = f'structures/bulk_structures/{atom_name}_oxide_supercell.lammps'

with open(lammps_path, 'w') as f:
    f.write('# Pure Ni bulk — written by Notebook 01\n\n')
    f.write(f'{len(supercell)} atoms\n')
    f.write(f'{len(MASSES)} atom types\n\n')
    f.write(f'0.0  {L[0]:.10f}  xlo xhi\n')
    f.write(f'0.0  {L[1]:.10f}  ylo yhi\n')
    f.write(f'0.0  {L[2]:.10f}  zlo zhi\n\n')
    f.write('Masses\n\n')
    for t, (m, el) in MASSES.items():
        f.write(f'{t}  {m}  # {el}\n')
    f.write('\nAtoms # atomic\n\n')
    for i, (sym, p) in enumerate(zip(syms, pos), 1):
        t = elem_to_type[sym]
        f.write(f'{i}  {t}  {p[0]:.10f}  {p[1]:.10f}  {p[2]:.10f}\n')

print(f'Written : {lammps_path}')
print(f'Total atoms : {len(supercell)}')
print(f'All atoms type  — all other types reserved for future use')

Written : structures/bulk_structures/Ni_oxide_supercell.lammps
Total atoms : 1000
All atoms type  — all other types reserved for future use
